# LivePortrait GStreamer Plugin Tutorial

This notebook demonstrates how to use the `gst-liveportrait` plugin for real-time head reenactment using our pre-built Docker environment.

## 1. Prerequisites

Ensure you have the TensorRT engines exported as described in the `README.md`. You will need Docker and NVIDIA Container Toolkit installed on your host machine.

## 2. Compile the Plugin

Even with the pre-built image, you must compile the C++ source code for your specific architecture inside the container.

In [ ]:
!docker run --rm -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest bash -c "mkdir -p build && cd build && cmake .. && make -j$(nproc)"

## 3. Run using the Python Wrapper (CLI)

The easiest way to run the plugin is using the `liveportrait_process.py` wrapper script. This script automatically pulls and uses the `ducksouplab/liveportrait_gst:latest` image.

In [ ]:
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/tutorial_output.mp4 \
    --source ../assets/test_image.jpg \
    --config ../checkpoints/

## 4. Dynamic Eye Retargeting

This advanced feature allows you to manually control eyelid closure and gaze. 

### Step 4.1: Download and Compile the Eye Engine
You need the official `stitching_eye.onnx` model.

In [ ]:
!wget https://huggingface.co/warmshao/FasterLivePortrait/resolve/main/liveportrait_onnx/stitching_eye.onnx -O ../stitching_eye.onnx
!docker run --rm --gpus all -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest python3 compile_trt.py

### Step 4.2: Run with Eye Controls
We use `--enable-eye-retargeting` and set `--eyes-open-ratio` to `0.0` (closed) with a `--eye-retargeting-strength` of `1.5` to ensure a deep blink.

In [ ]:
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/eye_retarget_test.mp4 \
    --source ../assets/test_image.jpg \
    --config ../checkpoints/ \
    --enable-eye-retargeting \
    --eyes-open-ratio 0.0 \
    --eye-retargeting-strength 1.5

## 5. Manual GStreamer Command

To run the raw GStreamer pipeline manually:

In [ ]:
!docker run --rm --gpus all -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest bash -c "\
    GST_PLUGIN_PATH=./build gst-launch-1.0 \
    filesrc location=assets/video_example.mp4 ! \
    decodebin ! videoconvert ! \
    videocrop left=280 right=280 ! \
    videoscale ! video/x-raw,width=512,height=512,format=RGB ! \
    liveportrait config-path=./checkpoints source-image=assets/test_image.jpg ! \
    videoconvert ! x264enc ! mp4mux ! filesink location=outputs/tutorial_manual_output.mp4"